In [ ]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random
from microlane.evalutation.lane_eval import LaneEval

Suppose we have two json files, one is the ground truth and the other is the prediction.

We assume that each line in prediction json file is corresponding to the ground truth json, which means both lines are related to the same image.

In [ ]:
prediction_json_path = "/home/suyog/desktop/projects/microlane/results/2026_04_16__15_03_37_batch_pipeline_testing_with_no_augmentation_/prediction.json"
gt_json_path = "/home/suyog/assets/datasets/TuSimple/test_label_new.json"

In [ ]:
with open(prediction_json_path, 'r') as f:
    content = f.read().strip()
    try:
        # Try loading as full JSON (list of dicts)
        json_pred = json.loads(content)
        if not isinstance(json_pred, list):
            raise Exception("Prediction JSON must be a list of objects.")
    except Exception:
        # Fallback: assume JSON lines format
        json_pred = [json.loads(line) for line in content.splitlines() if line.strip()]
    
json_gt = [json.loads(line) for line in open(gt_json_path)]

In [ ]:
randint = random.randint(0, 199)
pred, gt = json_pred[randint], json_gt[randint]
pred_lanes = pred['lanes']
run_time = pred['run_time']
gt_lanes = gt['lanes']
y_samples = gt['h_samples']
raw_file = gt['raw_file']


In [ ]:
img = plt.imread(raw_file)
plt.imshow(img)
plt.show()

In [ ]:
gt_lanes_vis = [[(x, y) for (x, y) in zip(lane, y_samples) if x >= 0] for lane in gt_lanes]
img_vis = img.copy()

for lane in gt_lanes_vis:
    for pt in lane:
        cv2.circle(img_vis, pt, radius=5, color=(0, 255, 0))

plt.imshow(img_vis)
plt.show()

In [ ]:
gt_lanes_vis = [[(x, y) for (x, y) in zip(lane, y_samples) if x >= 0] for lane in gt_lanes]
pred_lanes_vis = [[(x, y) for (x, y) in zip(lane, y_samples) if x >= 0] for lane in pred_lanes]

img_vis = img.copy()

# Draw GT lanes (green)
for lane in gt_lanes_vis:
    if len(lane) < 2:
        continue
    pts = np.array(lane, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(img_vis, [pts], isClosed=False, color=(0,255,0), thickness=5)

# Draw predicted lanes (red)
for lane in pred_lanes_vis:
    if len(lane) < 2:
        continue
    pts = np.array(lane, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(img_vis, [pts], isClosed=False, color=(0,0,255), thickness=2)

plt.imshow(img_vis)
plt.axis('off')
plt.show()

In [ ]:
np.random.shuffle(pred_lanes)
# Overall Accuracy, False Positive Rate, False Negative Rate
print(LaneEval.bench(pred_lanes, gt_lanes, y_samples, run_time))